In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [3]:
!pip install nvcc4jupyter

In [4]:
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmp0eobdecz".


In [6]:
%%cuda
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <chrono>
#include <math.h>
#include <utility>


// cuda kernel
// каждый поток вычисляет один элемент матр C
__global__ void matrixMulKernel(float* a, float* b, float* c, int rows_a, int cols_a, int cols_b)
{
  // индексы строки и столбца для текущего потока
  int row = blockIdx.y * blockDim.y + threadIdx.y;
  int col = blockIdx.x * blockDim.x + threadIdx.x;

  // если выход за границы матрицы
  if (row < rows_a && col < cols_b)
  {
    float value = 0.0f;
    // скалярное произведение строки матр A и столбца матр B
    for (int k = 0; k < cols_a; k++)
    {
      value += a[row * cols_a + k] * b[k * cols_b + col];
    }

    c[row * cols_b + col] = value;
  }
}


// CPU
// последовательное умножение матр
std::pair<float*, double> matrix_multiply_cpu(float* A, float* B, int rows_a, int cols_a, int cols_b)
{
  // выделение памяти
  float* C = (float*)malloc(rows_a * cols_b * sizeof(float));

  auto start = std::chrono::high_resolution_clock::now();
  for (int i = 0; i < rows_a; i++)
  {
    for (int j = 0; j < cols_b; j++)
    {
      C[i * cols_b + j] = 0.0f;
      for (int k = 0; k < cols_a; k++)
      {
        C[i * cols_b + j] += A[i * cols_a + k] * B[k * cols_b + j];
      }
    }
  }
  auto end = std::chrono::high_resolution_clock::now();
  double time_taken = std::chrono::duration<double>(end - start).count();

  return {C, time_taken};
}


// GPU
// параллельное умножение матриц
std::pair<float*, double> matrix_multiply_gpu(float* A, float* B, int rows_a, int cols_a, int cols_b)
{
  float* C = (float*)malloc(rows_a * cols_b * sizeof(float));
  float *A_d, *B_d, *C_d;
  size_t size_A = rows_a * cols_a * sizeof(float);
  size_t size_B = cols_a * cols_b * sizeof(float);
  size_t size_C = rows_a * cols_b * sizeof(float);

  auto start = std::chrono::high_resolution_clock::now();
  // выделение памяти на гпу
  cudaMalloc(&A_d, size_A);
  cudaMalloc(&B_d, size_B);
  cudaMalloc(&C_d, size_C);
  // копирование с цпу на гпу
  cudaMemcpy(A_d, A, size_A, cudaMemcpyHostToDevice);
  cudaMemcpy(B_d, B, size_B, cudaMemcpyHostToDevice);

  // каждый поток это один элемент матрицы
  dim3 block(16, 16);
  dim3 grid((cols_b + 15) / 16, (rows_a + 15) / 16);
  // запускаем ядро
  matrixMulKernel<<<grid, block>>>(A_d, B_d, C_d, rows_a, cols_a, cols_b);
  // для ожидания, синхрон
  cudaDeviceSynchronize();
  // копирование на цпу
  cudaMemcpy(C, C_d, size_C, cudaMemcpyDeviceToHost);

  cudaFree(A_d);
  cudaFree(B_d);
  cudaFree(C_d);

  auto end = std::chrono::high_resolution_clock::now();
  double time_taken = std::chrono::duration<double>(end - start).count();

  return {C, time_taken};
}


// сравнение цпу и гпу
bool check(float* A, float* B, int size)
{
  for (int i = 0; i < size; i++)
  {
      if (fabs(A[i] - B[i]) > 1e-3)
        return false;
  }

  return true;
}


int main()
{
  // размер матрицы вручную
  int N = 2000;
  size_t size = N * N * sizeof(float);

  // выделение памяти
  float* A = (float*)malloc(size);
  float* B = (float*)malloc(size);

  // рандом с числами
  for (int i = 0; i < N * N; i++)
  {
      A[i] = rand() % 10;
      B[i] = rand() % 10;
  }

  // GPU
  auto [C_gpu, time_gpu] = matrix_multiply_gpu(A, B, N, N, N);
  printf("GPU /// TIME: %f sec\n", time_gpu);
  // CPU
  auto [C_cpu, time_cpu] = matrix_multiply_cpu(A, B, N, N, N);
  printf("CPU /// TIME: %f sec\n", time_cpu);

  bool ok = check(C_cpu, C_gpu, N * N);
  printf("correct? %s\n", ok ? "TRUE" : "FALSE");
  printf("speedup: %f\n", time_cpu / time_gpu);

  free(A);
  free(B);
  free(C_cpu);
  free(C_gpu);

  return 0;
}

GPU /// TIME: 0.354544 sec
CPU /// TIME: 97.294784 sec
correct? TRUE
speedup: 274.422017

